# Section 2.7 Sensitivity Analysis and Stress Testing

This notebook runs and inspects OW sensitivity/stress outputs. It uses clock-time signal delays and stock/date grouped scenario logic.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / "outputs" / "stress"
FIG_DIR = OUT_DIR / "figures"
ALPHA_INPUT = PROJECT_ROOT / "outputs" / "alphas" / "strategy_alpha_input_h5m_rho010_H5m.csv"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALPHA_INPUT exists:", ALPHA_INPUT.exists())
print("OUT_DIR:", OUT_DIR)

PROJECT_ROOT: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project
ALPHA_INPUT exists: True
OUT_DIR: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/stress


In [2]:
from src.ow_strategy import attach_scaling_factors
from src.scaling_factors import compute_or_load_scaling_factors
from src.strategy_config import OWStrategyConfig
from src.stress_config import SensitivityConfig, StressTestConfig, StressRunConfig
from src.stress_tests import load_alpha_input, run_all_stress_tests
from src.stress_validation import validate_stress_outputs, save_stress_validation_report

## 1. Baseline Strategy

In [3]:
alpha = load_alpha_input(ALPHA_INPUT)
scaling_path = PROJECT_ROOT / "outputs" / "scaling" / "scaling_factors_20d.csv"
scaling = compute_or_load_scaling_factors(PROJECT_ROOT / "data" / "binSamples", scaling_path, force_recompute=False)
base_ow_config = OWStrategyConfig(
    scaling_factors_path=str(scaling_path.relative_to(PROJECT_ROOT)),
    use_scaling_factors=True,
    impact_lambda=1.0,
    impact_half_life_minutes=5.0,
)
alpha = attach_scaling_factors(alpha, scaling, base_ow_config)
print("Rows:", len(alpha))
print("Scaling coverage:", alpha["has_scaling"].mean())
display(alpha.head())

Rows: 200000
Scaling coverage: 0.0


,date,time,timestamp,stock,mid,alpha_raw,alpha_for_strategy,future_return_h,valid_future_return,spread,depth,source_file,sigma,ADV,scaling_window_days,has_full_scaling_window,px_vol_current_day,daily_volume_current_day,has_scaling
0,2019-01-02,09:30:10,2019-01-02 09:30:10,A,66.215,-0.000048,-0.000048,0.003020,True,0.285,300.0,bin201901.csv,NaN,NaN,0,False,0.000393,217804.0,False
1,2019-01-02,09:30:20,2019-01-02 09:30:20,A,66.335,-0.000114,-0.000050,0.001583,True,0.145,350.0,bin201901.csv,NaN,NaN,0,False,0.000393,217804.0,False
2,2019-01-02,09:30:30,2019-01-02 09:30:30,A,66.335,-0.000123,-0.000052,0.001583,True,0.145,600.0,bin201901.csv,NaN,NaN,0,False,0.000393,217804.0,False
3,2019-01-02,09:30:40,2019-01-02 09:30:40,A,66.340,-0.000239,-0.000056,0.001733,True,0.140,500.0,bin201901.csv,NaN,NaN,0,False,0.000393,217804.0,False
4,2019-01-02,09:31:00,2019-01-02 09:31:00,A,66.270,0.000156,-0.000046,0.003320,True,0.070,375.0,bin201901.csv,NaN,NaN,0,False,0.000393,217804.0,False


## 2. Run or Load Stress Outputs

In [4]:
sensitivity_config = SensitivityConfig()
stress_config = StressTestConfig()
run_config = StressRunConfig(output_dir=str(OUT_DIR))

# Rerun scenarios. Comment this cell out if you only want to inspect existing CSVs.
sensitivity_summary, stress_summary = run_all_stress_tests(alpha, base_ow_config, sensitivity_config, stress_config, run_config)
checks = validate_stress_outputs(OUT_DIR)
save_stress_validation_report(checks, OUT_DIR)
display(sensitivity_summary.head())
display(stress_summary)

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,impact_lambda_multiplier,impact_half_life_minutes,baseline_total_net_pnl,delta_net_pnl_vs_baseline,pct_net_pnl_degradation_vs_baseline,delta_sharpe_vs_baseline,delta_turnover_vs_baseline,delta_impact_cost_vs_baseline,delta_max_drawdown_vs_baseline,delta_max_abs_impact_vs_baseline
0,baseline_OW,baseline,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,sensitivity_lambda_mult_0.5_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,0.5,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
2,sensitivity_lambda_mult_1_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
3,sensitivity_lambda_mult_2_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,2.0,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
4,sensitivity_impact_H1m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,1.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,true_total_net_pnl,model_error_cost,baseline_total_net_pnl,delta_net_pnl_vs_baseline,pct_net_pnl_degradation_vs_baseline,delta_sharpe_vs_baseline,delta_turnover_vs_baseline,delta_impact_cost_vs_baseline,delta_max_drawdown_vs_baseline,delta_max_abs_impact_vs_baseline
0,baseline_OW,baseline,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,delay_1m_OW,signal_delay_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
2,forced_liq_1200_stop_OW,forced_liquidation_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
3,wrong_impact_lambda2_H30_OW,wrong_impact_parameter_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


## 3. Sensitivity Analysis

In [5]:
sensitivity = pd.read_csv(OUT_DIR / "sensitivity_summary.csv")
display(sensitivity)
for fig in ["impact_lambda_sensitivity.png", "impact_half_life_sensitivity.png", "alpha_decay_sensitivity.png"]:
    p = FIG_DIR / fig
    print(fig, p.exists())

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,impact_lambda_multiplier,impact_half_life_minutes,baseline_total_net_pnl,delta_net_pnl_vs_baseline,pct_net_pnl_degradation_vs_baseline,delta_sharpe_vs_baseline,delta_turnover_vs_baseline,delta_impact_cost_vs_baseline,delta_max_drawdown_vs_baseline,delta_max_abs_impact_vs_baseline
0,baseline_OW,baseline,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,sensitivity_lambda_mult_0.5_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,0.5,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
2,sensitivity_lambda_mult_1_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
3,sensitivity_lambda_mult_2_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,2.0,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
4,sensitivity_impact_H1m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,1.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
5,sensitivity_impact_H5m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,5.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
6,sensitivity_impact_H30m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,30.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
7,sensitivity_impact_H60m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,1.0,60.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


impact_lambda_sensitivity.png True
impact_half_life_sensitivity.png True
alpha_decay_sensitivity.png False


## 4. Signal Delay Stress

In [6]:
delay_summary = pd.read_csv(OUT_DIR / "signal_delay_summary.csv") if (OUT_DIR / "signal_delay_summary.csv").exists() else pd.DataFrame()
display(delay_summary)
delayed = pd.read_csv(OUT_DIR / "signal_delay_trades.csv", nrows=10) if (OUT_DIR / "signal_delay_trades.csv").exists() else pd.DataFrame()
display(delayed[[c for c in delayed.columns if c in ["timestamp", "stock", "alpha", "alpha_delayed_1m", "delayed_alpha_age_seconds", "net_pnl"]]].head())

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,max_abs_position,max_abs_impact,max_participation_rate,mean_participation_rate,n_trades,n_rows,n_stock_days,n_dates,n_stocks,signal_delay_minutes
0,delay_1m_OW,signal_delay_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,0.0,0.0,NaN,NaN,0,200000,90,2,50,1.0


,timestamp,stock,alpha,net_pnl,alpha_delayed_1m,delayed_alpha_age_seconds
0,2019-01-02 09:30:10,A,0.0,0.0,0.0,NaN
1,2019-01-02 09:30:20,A,0.0,0.0,0.0,NaN
2,2019-01-02 09:30:30,A,0.0,0.0,0.0,NaN
3,2019-01-02 09:30:40,A,0.0,0.0,0.0,NaN
4,2019-01-02 09:31:00,A,0.0,-0.0,0.0,NaN


## 5. Forced Liquidation Stress

In [7]:
forced_summary = pd.read_csv(OUT_DIR / "forced_liquidation_summary.csv") if (OUT_DIR / "forced_liquidation_summary.csv").exists() else pd.DataFrame()
display(forced_summary)
forced = pd.read_csv(OUT_DIR / "forced_liquidation_trades.csv", usecols=lambda c: c in ["date", "time", "stock", "is_forced_liquidation", "forced_liquidation_trade", "position_after", "net_pnl"]) if (OUT_DIR / "forced_liquidation_trades.csv").exists() else pd.DataFrame()
display(forced[forced.get("is_forced_liquidation", False).astype(bool)].head())

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,n_rows,n_stock_days,n_dates,n_stocks,forced_liquidation_time,resume_after_liquidation,number_of_liquidation_events,mean_abs_liquidation_trade,max_abs_liquidation_trade,liquidation_cost_total
0,forced_liq_1200_stop_OW,forced_liquidation_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,200000,90,2,50,12:00:00,False,90,0.0,0.0,0.0


,date,time,stock,position_after,net_pnl,is_forced_liquidation,forced_liquidation_trade
846,2019-01-02,12:00:00,A,0.0,-0.0,True,-0.0
3113,2019-01-03,12:00:00,A,0.0,-0.0,True,-0.0
5438,2019-01-02,12:00:00,AAL,0.0,-0.0,True,-0.0
7776,2019-01-03,12:00:00,AAL,0.0,-0.0,True,-0.0
10020,2019-01-02,12:00:00,AAP,0.0,0.0,True,-0.0


## 6. Wrong Impact Parameter Stress

In [8]:
wrong_summary = pd.read_csv(OUT_DIR / "wrong_impact_summary.csv") if (OUT_DIR / "wrong_impact_summary.csv").exists() else pd.DataFrame()
display(wrong_summary)

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,n_stock_days,n_dates,n_stocks,assumed_lambda,true_lambda,assumed_impact_half_life_minutes,true_impact_half_life_minutes,assumed_total_net_pnl,true_total_net_pnl,model_error_cost
0,wrong_impact_lambda2_H30_OW,wrong_impact_parameter_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,90,2,50,1.0,2.0,5.0,30.0,0.0,0.0,0.0


## 7. Scenario Comparison

In [9]:
all_summary = pd.read_csv(OUT_DIR / "all_scenarios_summary.csv")
display(all_summary.sort_values("total_net_pnl"))
plot_files = sorted(FIG_DIR.glob("*.png"))
for p in plot_files:
    print(p.relative_to(PROJECT_ROOT))

,scenario_name,scenario_type,total_gross_pnl,total_net_pnl,total_signed_impact_cost,total_quadratic_impact_cost,mean_daily_net_pnl,std_daily_net_pnl,daily_sharpe,annualized_sharpe,...,mean_abs_liquidation_trade,max_abs_liquidation_trade,liquidation_cost_total,assumed_lambda,true_lambda,assumed_impact_half_life_minutes,true_impact_half_life_minutes,assumed_total_net_pnl,true_total_net_pnl,model_error_cost
0,baseline_OW,baseline,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sensitivity_lambda_mult_0.5_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sensitivity_lambda_mult_1_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sensitivity_lambda_mult_2_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sensitivity_impact_H1m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sensitivity_impact_H5m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,sensitivity_impact_H30m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,sensitivity_impact_H60m_OW,impact_parameter_sensitivity,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,delay_1m_OW,signal_delay_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,forced_liq_1200_stop_OW,forced_liquidation_stress,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


outputs/stress/figures/baseline_vs_stress_cumulative_wealth.png
outputs/stress/figures/forced_liquidation_wealth.png
outputs/stress/figures/impact_half_life_sensitivity.png
outputs/stress/figures/impact_lambda_sensitivity.png
outputs/stress/figures/scenario_drawdown_bar.png
outputs/stress/figures/scenario_net_pnl_bar.png
outputs/stress/figures/scenario_sharpe_bar.png
outputs/stress/figures/signal_delay_wealth.png
outputs/stress/figures/wrong_impact_wealth.png


## 8. Validation

In [10]:
checks = validate_stress_outputs(OUT_DIR)
checks_df = pd.DataFrame.from_dict(checks, orient="index")
display(checks_df)
report = OUT_DIR / "stress_validation_report.txt"
print(report.read_text() if report.exists() else "validation report missing")

,status,message
required_summary_files,PASS,missing=[]
baseline_exists,PASS,baseline_OW present
unique_scenario_names,PASS,duplicates=0
key_metrics_present,PASS,finite_metric_share=80.00%
delayed_scenario_exists,PASS,signal_delay_stress present=True
forced_liquidation_scenario_exists,PASS,forced_liquidation_stress present=True
wrong_impact_scenario_exists,PASS,wrong_impact_parameter_stress present=True
baseline_comparison_columns,PASS,"comparison_cols=['delta_net_pnl_vs_baseline', ..."
zero_baseline_pct_degradation,WARN,baseline net pnl is near zero; pct degradation...
skipped_scenarios,WARN,skipped_scenarios.csv exists


Section 2.7 Stress Testing Validation Report

Checks:
- required_summary_files: PASS - missing=[]
- baseline_exists: PASS - baseline_OW present
- unique_scenario_names: PASS - duplicates=0
- key_metrics_present: PASS - finite_metric_share=80.00%
- delayed_scenario_exists: PASS - signal_delay_stress present=True
- forced_liquidation_scenario_exists: PASS - forced_liquidation_stress present=True
- wrong_impact_scenario_exists: PASS - wrong_impact_parameter_stress present=True
- baseline_comparison_columns: PASS - comparison_cols=['delta_net_pnl_vs_baseline', 'pct_net_pnl_degradation_vs_baseline', 'delta_sharpe_vs_baseline', 'delta_turnover_vs_baseline', 'delta_impact_cost_vs_baseline', 'delta_max_drawdown_vs_baseline', 'delta_max_abs_impact_vs_baseline']
- zero_baseline_pct_degradation: WARN - baseline net pnl is near zero; pct degradation should be NaN
- skipped_scenarios: WARN - skipped_scenarios.csv exists
- plots_exist: PASS - n_figures=9
- forced_liquidation_events: PASS - events=90

## 9. Report-Ready Takeaways

In [11]:
takeaways = [
    "Rho/horizon sensitivity is skipped unless scenario alpha files are generated for those grids.",
    "Alpha decay sensitivity runs only when alpha_state_HXm columns are present in the strategy input.",
    "Impact lambda and impact half-life sensitivities are implemented for OW.",
    "Signal delay is clock-time based: last alpha at or before t minus the delay.",
    "Forced liquidation uses a block trade at the first timestamp at or after 12:00 per stock/day.",
    "Wrong-impact stress keeps the assumed trade path fixed and recomputes wealth under true OW parameters.",
    "Current outputs may be zero because early-January alpha rows have no previous 20-day scaling coverage.",
    "Final report should rerun on the selected out-of-sample period with calibrated parameters.",
]
for item in takeaways:
    print("-", item)

- Rho/horizon sensitivity is skipped unless scenario alpha files are generated for those grids.
- Alpha decay sensitivity runs only when alpha_state_HXm columns are present in the strategy input.
- Impact lambda and impact half-life sensitivities are implemented for OW.
- Signal delay is clock-time based: last alpha at or before t minus the delay.
- Forced liquidation uses a block trade at the first timestamp at or after 12:00 per stock/day.
- Wrong-impact stress keeps the assumed trade path fixed and recomputes wealth under true OW parameters.
- Current outputs may be zero because early-January alpha rows have no previous 20-day scaling coverage.
- Final report should rerun on the selected out-of-sample period with calibrated parameters.
